# Ensembl REST

![Ensembl REST](https://proto-bio.github.io/proto-assets/images/tool/ensembl/hero.png)

Five tools share the `ensembl` toolkit, one per endpoint family:

- **`ensembl-lookup`** — gene lookup by Ensembl gene ID **or** gene symbol.
- **`ensembl-sequence`** — DNA / cDNA / CDS / protein sequence retrieval by Ensembl ID.
- **`ensembl-overlap`** — features overlapping a region (genes, exons, regulatory elements, motifs, variants).
- **`ensembl-xrefs`** — cross-references from an Ensembl ID to external databases.
- **`ensembl-vep`** — variant-effect prediction from HGVS notation.

All five call `rest.ensembl.org` (GRCh38) by default; `assembly="GRCh37"` routes to `grch37.rest.ensembl.org`.

In [1]:
from proto_tools.tools.database_retrieval import (
    EnsemblLookupConfig,
    EnsemblLookupInput,
    EnsemblOverlapConfig,
    EnsemblOverlapInput,
    EnsemblSequenceConfig,
    EnsemblSequenceInput,
    EnsemblVEPInput,
    EnsemblXrefsInput,
    run_ensembl_lookup,
    run_ensembl_overlap,
    run_ensembl_sequence,
    run_ensembl_vep,
    run_ensembl_xrefs,
)

In [2]:
from proto_tools.utils.notebook_docs import display_api_reference

## ensembl-lookup — gene lookup with transcript expansion

In [3]:
# Display input docs
display_api_reference("ensembl", "input", "run_ensembl_lookup")

**Input** — `EnsemblLookupInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>ensembl_id</code> | <code>str &#124; None</code> | <code>None</code> | Ensembl gene ID (e.g. ENSG00000012048) |
| <code>symbol</code> | <code>str &#124; None</code> | <code>None</code> | Gene symbol (e.g. BRCA1) |

In [4]:
# Display config docs
display_api_reference("ensembl", "config", "run_ensembl_lookup")

**Config** — `EnsemblLookupConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>species</code> | <code>Literal['homo_sapiens', 'mus_musculus', 'rattus_norvegicus', 'danio_rerio', 'saccharomyces_cerevisiae']</code> | <code>'homo_sapiens'</code> | Species (used when symbol is provided) |
| <code>assembly</code> | <code>Literal['GRCh38', 'GRCh37']</code> | <code>'GRCh38'</code> | Genome assembly; GRCh37 routes to grch37.rest.ensembl.org |
| <code>expand</code> | <code>bool</code> | <code>False</code> | Include transcripts, translations, and exons in the response |
| <code>mane</code> | <code>bool</code> | <code>False</code> | Include MANE Select annotations (lookup-by-id only; requires expand=True) |
| <code>phenotypes</code> | <code>bool</code> | <code>False</code> | Include phenotype annotations on gene records (lookup-by-id only) |
| <code>utr</code> | <code>bool</code> | <code>False</code> | Include UTR coordinates per transcript (lookup-by-id only; requires expand=True) |

In [5]:
# Display output docs
display_api_reference("ensembl", "output", "run_ensembl_lookup")

**Output** — `EnsemblLookupOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>result</code> | <code>EnsemblGene</code> | required | The looked-up gene record |
| <code>source_url</code> | <code>str</code> | required | Final Ensembl REST URL that was hit |
| <code>raw_payload</code> | <code>dict[str, Any]</code> | <code>{}</code> | Raw API JSON returned by Ensembl |

**`EnsemblGene`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>id</code> | <code>str</code> | required | Stable Ensembl gene identifier (ENSG...) |
| <code>display_name</code> | <code>str &#124; None</code> | <code>None</code> | Human-readable gene symbol (e.g. 'BRCA1') |
| <code>description</code> | <code>str &#124; None</code> | <code>None</code> | Free-text gene description from Ensembl |
| <code>biotype</code> | <code>str</code> | required | Gene biotype (protein_coding, lncRNA, ...) |
| <code>species</code> | <code>str</code> | required | Species slug (e.g. 'homo_sapiens') |
| <code>seq_region_name</code> | <code>str</code> | required | Chromosome or contig name |
| <code>start</code> | <code>int</code> | required | 1-indexed inclusive genomic start coordinate |
| <code>end</code> | <code>int</code> | required | 1-indexed inclusive genomic end coordinate |
| <code>strand</code> | <code>int</code> | required | Genomic strand (+1 or -1) |
| <code>assembly_name</code> | <code>str</code> | required | Genome assembly name (e.g. 'GRCh38') |
| <code>canonical_transcript</code> | <code>str &#124; None</code> | <code>None</code> | Canonical transcript ID with version |
| <code>Transcript</code> | <code>list[EnsemblTranscript]</code> | <code>[]</code> | Transcript records (empty unless expand=True) |

**`EnsemblTranscript`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>id</code> | <code>str</code> | required | Stable Ensembl transcript identifier (ENST...) |
| <code>display_name</code> | <code>str &#124; None</code> | <code>None</code> | Human-readable transcript name, if any |
| <code>biotype</code> | <code>str</code> | required | Transcript biotype (protein_coding, lncRNA, ...) |
| <code>is_canonical</code> | <code>bool</code> | <code>False</code> | Whether this is the canonical transcript |
| <code>start</code> | <code>int</code> | required | 1-indexed inclusive genomic start coordinate |
| <code>end</code> | <code>int</code> | required | 1-indexed inclusive genomic end coordinate |
| <code>strand</code> | <code>int</code> | required | Genomic strand (+1 or -1) |
| <code>seq_region_name</code> | <code>str</code> | required | Chromosome or contig name |
| <code>assembly_name</code> | <code>str</code> | required | Genome assembly name (e.g. 'GRCh38') |
| <code>Exon</code> | <code>list[EnsemblExon]</code> | <code>[]</code> | Exon records (empty unless expand=True) |
| <code>Translation</code> | <code>EnsemblTranslation &#124; None</code> | <code>None</code> | Protein translation; None for non-coding biotypes |

**`EnsemblExon`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>id</code> | <code>str</code> | required | Stable Ensembl exon identifier (ENSE...) |
| <code>seq_region_name</code> | <code>str</code> | required | Chromosome or contig name (e.g. '17') |
| <code>start</code> | <code>int</code> | required | 1-indexed inclusive genomic start coordinate |
| <code>end</code> | <code>int</code> | required | 1-indexed inclusive genomic end coordinate |
| <code>strand</code> | <code>int</code> | required | Genomic strand (+1 or -1) |
| <code>assembly_name</code> | <code>str</code> | required | Genome assembly name (e.g. 'GRCh38') |
| <code>version</code> | <code>int &#124; None</code> | <code>None</code> | Ensembl version of this exon |

**`EnsemblTranslation`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>id</code> | <code>str</code> | required | Stable Ensembl protein identifier (ENSP...) |
| <code>start</code> | <code>int</code> | required | 1-indexed inclusive start of the CDS on the parent transcript |
| <code>end</code> | <code>int</code> | required | 1-indexed inclusive end of the CDS on the parent transcript |
| <code>length</code> | <code>int</code> | required | Protein length in amino acids |
| <code>Parent</code> | <code>str</code> | required | Parent transcript ID (ENST...) |

In [6]:
out = run_ensembl_lookup(EnsemblLookupInput(symbol="BRCA1"), EnsemblLookupConfig(expand=True))
assert out.success
gene = out.result
print(f"{gene.display_name} ({gene.id}): {len(gene.Transcript)} transcripts; canonical={gene.canonical_transcript}")

BRCA1 (ENSG00000012048): 59 transcripts; canonical=ENST00000357654.9


## ensembl-sequence — protein sequence of the canonical transcript

In [7]:
# Display input docs
display_api_reference("ensembl", "input", "run_ensembl_sequence")

**Input** — `EnsemblSequenceInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>ensembl_id</code> | <code>str</code> | required | Ensembl ID (ENSG..., ENST..., ENSP...) |

In [8]:
# Display config docs
display_api_reference("ensembl", "config", "run_ensembl_sequence")

**Config** — `EnsemblSequenceConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>sequence_type</code> | <code>Literal['genomic', 'cdna', 'cds', 'protein']</code> | <code>'genomic'</code> | genomic = DNA+UTR+introns; cdna = mRNA+UTR; cds = coding only; protein = translation |
| <code>assembly</code> | <code>Literal['GRCh38', 'GRCh37']</code> | <code>'GRCh38'</code> | Genome assembly; GRCh37 routes to grch37.rest.ensembl.org |
| <code>multiple_sequences</code> | <code>bool</code> | <code>False</code> | Return all sequences for IDs that map to multiple records |
| <code>mask</code> | <code>Literal['hard', 'soft'] &#124; None</code> | <code>None</code> | Mask repeats: 'hard' (N) or 'soft' (lowercase); genomic only |
| <code>mask_feature</code> | <code>bool</code> | <code>False</code> | Mask introns (genomic) or UTRs (cdna) so the primary feature stands out |
| <code>expand_3prime</code> | <code>int &#124; None</code> | <code>None</code> | Add bases to the 3' end (genomic only; incompatible with 'end') |
| <code>expand_5prime</code> | <code>int &#124; None</code> | <code>None</code> | Add bases to the 5' end (genomic only; incompatible with 'start') |
| <code>start</code> | <code>int &#124; None</code> | <code>None</code> | 1-indexed start trim coordinate (incompatible with expand_5prime) |
| <code>end</code> | <code>int &#124; None</code> | <code>None</code> | 1-indexed end trim coordinate (incompatible with expand_3prime) |

In [9]:
# Display output docs
display_api_reference("ensembl", "output", "run_ensembl_sequence")

**Output** — `EnsemblSequenceOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[EnsemblSequence]</code> | <code>[]</code> | Fetched sequence record(s) |
| <code>source_url</code> | <code>str</code> | required | Final Ensembl REST URL that was hit |
| <code>raw_payload</code> | <code>list[dict[str, Any]]</code> | <code>[]</code> | Raw API JSON returned by Ensembl |

**`EnsemblSequence`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>id</code> | <code>str</code> | required | Stable Ensembl ID echoed by the server (may differ from input) |
| <code>desc</code> | <code>str &#124; None</code> | <code>None</code> | Description string returned by the server |
| <code>mol_type</code> | <code>str &#124; None</code> | <code>None</code> | Molecule type ('dna', 'protein', ...) |
| <code>seq</code> | <code>str</code> | required | Raw sequence string (DNA, cDNA, CDS, or protein) |

In [10]:
canonical_id = gene.canonical_transcript.split(".")[0]
seq_out = run_ensembl_sequence(
    EnsemblSequenceInput(ensembl_id=canonical_id),
    EnsemblSequenceConfig(sequence_type="protein"),
)
print(f"{seq_out.results[0].id}: {len(seq_out.results[0].seq)} aa")

ENSP00000350283: 1863 aa


## ensembl-overlap — regulatory features at the BRCA1 locus

In [11]:
# Display input docs
display_api_reference("ensembl", "input", "run_ensembl_overlap")

**Input** — `EnsemblOverlapInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>ensembl_id</code> | <code>str</code> | required | Ensembl ID (ENSG..., ENST..., ENSP...) |

In [12]:
# Display config docs
display_api_reference("ensembl", "config", "run_ensembl_overlap")

**Config** — `EnsemblOverlapConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>overlap_feature</code> | <code>Literal['band', 'gene', 'transcript', 'cds', 'exon', 'repeat', 'simple', 'misc', 'variation', 'somatic_variation', 'structural_variation', 'somatic_structural_variation', 'constrained', 'regulatory', 'motif', 'mane']</code> | <code>'gene'</code> | Type of feature to retrieve (e.g. gene, transcript, exon, regulatory, variation) |
| <code>assembly</code> | <code>Literal['GRCh38', 'GRCh37']</code> | <code>'GRCh38'</code> | Genome assembly; GRCh37 routes to grch37.rest.ensembl.org |
| <code>biotype</code> | <code>Literal['protein_coding', 'lncRNA', 'lincRNA', 'miRNA', 'snRNA', 'snoRNA', 'rRNA', 'Mt_rRNA', 'Mt_tRNA', 'tRNA', 'misc_RNA', 'ribozyme', 'scaRNA', 'sRNA', 'vault_RNA', 'Y_RNA', 'antisense_RNA', 'non_coding', 'processed_transcript', 'pseudogene', 'processed_pseudogene', 'transcribed_processed_pseudogene', 'transcribed_unitary_pseudogene', 'transcribed_unprocessed_pseudogene', 'unitary_pseudogene', 'unprocessed_pseudogene', 'translated_processed_pseudogene', 'translated_unprocessed_pseudogene', 'TR_C_gene', 'TR_D_gene', 'TR_J_gene', 'TR_V_gene', 'TR_J_pseudogene', 'TR_V_pseudogene', 'IG_C_gene', 'IG_D_gene', 'IG_J_gene', 'IG_V_gene', 'IG_C_pseudogene', 'IG_J_pseudogene', 'IG_V_pseudogene', 'IG_pseudogene', 'TEC', 'nonsense_mediated_decay', 'non_stop_decay', 'retained_intron'] &#124; None</code> | <code>None</code> | Restrict to a biotype (e.g. 'protein_coding'); most useful for gene/transcript features |
| <code>so_term</code> | <code>Literal['transcript_ablation', 'splice_acceptor_variant', 'splice_donor_variant', 'stop_gained', 'frameshift_variant', 'stop_lost', 'start_lost', 'transcript_amplification', 'feature_elongation', 'feature_truncation', 'inframe_insertion', 'inframe_deletion', 'missense_variant', 'protein_altering_variant', 'splice_region_variant', 'splice_donor_5th_base_variant', 'splice_donor_region_variant', 'splice_polypyrimidine_tract_variant', 'incomplete_terminal_codon_variant', 'start_retained_variant', 'stop_retained_variant', 'synonymous_variant', 'coding_sequence_variant', 'mature_miRNA_variant', '5_prime_UTR_variant', '3_prime_UTR_variant', 'non_coding_transcript_exon_variant', 'intron_variant', 'NMD_transcript_variant', 'non_coding_transcript_variant', 'coding_transcript_variant', 'upstream_gene_variant', 'downstream_gene_variant', 'TFBS_ablation', 'TFBS_amplification', 'TF_binding_site_variant', 'regulatory_region_ablation', 'regulatory_region_amplification', 'regulatory_region_variant', 'intergenic_variant', 'sequence_variant'] &#124; None</code> | <code>None</code> | Restrict variation features by SO consequence (e.g. 'missense_variant') |
| <code>variant_set</code> | <code>str &#124; None</code> | <code>None</code> | Restrict variation features to a named variant set (e.g. 'ClinVar') |

In [13]:
# Display output docs
display_api_reference("ensembl", "output", "run_ensembl_overlap")

**Output** — `EnsemblOverlapOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>result</code> | <code>list[EnsemblOverlapFeatureRecord]</code> | <code>[]</code> | Overlapping features (typed common fields + raw dict) |
| <code>source_url</code> | <code>str</code> | required | Final Ensembl REST URL that was hit |
| <code>raw_payload</code> | <code>list[dict[str, Any]]</code> | <code>[]</code> | Raw API JSON returned by Ensembl |

**`EnsemblOverlapFeatureRecord`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>feature_type</code> | <code>str</code> | required | Feature type ('gene', 'transcript', 'regulatory', 'variation', ...) |
| <code>id</code> | <code>str &#124; None</code> | <code>None</code> | Feature identifier where the API returns one |
| <code>biotype</code> | <code>str &#124; None</code> | <code>None</code> | Feature biotype where applicable |
| <code>start</code> | <code>int</code> | required | 1-indexed inclusive genomic start coordinate |
| <code>end</code> | <code>int</code> | required | 1-indexed inclusive genomic end coordinate |
| <code>strand</code> | <code>int</code> | required | Genomic strand (+1, -1, or 0 for unstranded) |
| <code>seq_region_name</code> | <code>str</code> | required | Chromosome or contig name |
| <code>raw</code> | <code>dict[str, Any]</code> | <code>{}</code> | Full upstream record for feature-specific fields |

In [14]:
overlap_out = run_ensembl_overlap(
    EnsemblOverlapInput(ensembl_id=gene.id),
    EnsemblOverlapConfig(overlap_feature="regulatory"),
)
print(f"{len(overlap_out.result)} regulatory features overlapping {gene.display_name}:")
for record in overlap_out.result[:5]:
    print(f"  {record.feature_type} {record.id} ({record.seq_region_name}:{record.start}-{record.end})")

11 regulatory features overlapping BRCA1:
  regulatory ENSR17_9M4PM (17:43084519-43084965)
  regulatory ENSR17_5D5LMG (17:43084749-43084770)
  regulatory ENSR17_5D5LRR (17:43085679-43085700)
  regulatory ENSR17_BBK45 (17:43124357-43125289)
  regulatory ENSR17_5D5TK5 (17:43125063-43125084)


## ensembl-xrefs — cross-reference Ensembl → UniProt

In [15]:
# Display input docs
display_api_reference("ensembl", "input", "run_ensembl_xrefs")

**Input** — `EnsemblXrefsInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>ensembl_id</code> | <code>str</code> | required | Ensembl ID (ENSG..., ENST..., ENSP...) |

In [16]:
# Display config docs
display_api_reference("ensembl", "config", "run_ensembl_xrefs")

**Config** — `EnsemblXrefsConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>assembly</code> | <code>Literal['GRCh38', 'GRCh37']</code> | <code>'GRCh38'</code> | Genome assembly; GRCh37 routes to grch37.rest.ensembl.org |
| <code>all_levels</code> | <code>bool</code> | <code>False</code> | Fan out xrefs to child transcripts and translations (gene queries) |
| <code>external_db</code> | <code>Literal['UniProtKB/Swiss-Prot', 'UniProtKB/TrEMBL', 'UniProtKB_all', 'HGNC', 'MGI', 'RGD', 'ZFIN', 'EntrezGene', 'RefSeq_mRNA', 'RefSeq_mRNA_predicted', 'RefSeq_ncRNA', 'RefSeq_ncRNA_predicted', 'RefSeq_peptide', 'RefSeq_peptide_predicted', 'RefSeq_genomic', 'CCDS', 'Pfam', 'InterPro', 'GO', 'PDB', 'STRING', 'Reactome', 'ArrayExpress', 'WikiGene', 'MIM_GENE', 'MIM_MORBID', 'ClinVar'] &#124; None</code> | <code>None</code> | Restrict to one external DB (e.g. 'UniProtKB/Swiss-Prot', 'HGNC') |
| <code>object_type</code> | <code>Literal['gene', 'transcript', 'translation'] &#124; None</code> | <code>None</code> | Restrict to one feature type when the stable ID resolves ambiguously |

In [17]:
# Display output docs
display_api_reference("ensembl", "output", "run_ensembl_xrefs")

**Output** — `EnsemblXrefsOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>result</code> | <code>list[EnsemblXref]</code> | <code>[]</code> | Cross-reference records to external databases |
| <code>source_url</code> | <code>str</code> | required | Final Ensembl REST URL that was hit |
| <code>raw_payload</code> | <code>list[dict[str, Any]]</code> | <code>[]</code> | Raw API JSON returned by Ensembl |

**`EnsemblXref`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>dbname</code> | <code>str</code> | required | External database name (e.g. 'Uniprot_gn', 'EntrezGene') |
| <code>db_display_name</code> | <code>str &#124; None</code> | <code>None</code> | Human-readable external database name |
| <code>display_id</code> | <code>str</code> | required | Display identifier in the external database |
| <code>primary_id</code> | <code>str</code> | required | Primary identifier in the external database |
| <code>description</code> | <code>str &#124; None</code> | <code>None</code> | External-DB description for this entry |
| <code>info_type</code> | <code>str &#124; None</code> | <code>None</code> | Cross-reference type (e.g. 'DIRECT', 'DEPENDENT') |

In [18]:
# Cross-reference the canonical protein to its reviewed Swiss-Prot accession (BRCA1 -> P38398).
protein_id = seq_out.results[0].id
xrefs_out = run_ensembl_xrefs(EnsemblXrefsInput(ensembl_id=protein_id))
swissprot = next(x for x in xrefs_out.result if x.dbname == "Uniprot/SWISSPROT")
print(f"Swiss-Prot accession for {gene.display_name}: {swissprot.primary_id} (via {protein_id})")

Swiss-Prot accession for BRCA1: P38398 (via ENSP00000350283)


## ensembl-vep — Variant Effect Prediction

HGVS forms supported: genomic (`9:g.22125504G>C`), coding (`ENST00000357654:c.5074G>A`), or protein (`ENSP00000418960:p.Tyr124Cys`).

In [19]:
# Display input docs
display_api_reference("ensembl", "input", "run_ensembl_vep")

**Input** — `EnsemblVEPInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>hgvs</code> | <code>str</code> | required | HGVS notation (genomic / coding / protein) |

In [20]:
# Display config docs
display_api_reference("ensembl", "config", "run_ensembl_vep")

**Config** — `EnsemblVEPConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>species</code> | <code>Literal['homo_sapiens', 'mus_musculus', 'rattus_norvegicus', 'danio_rerio', 'saccharomyces_cerevisiae']</code> | <code>'homo_sapiens'</code> | Species slug for VEP |
| <code>assembly</code> | <code>Literal['GRCh38', 'GRCh37']</code> | <code>'GRCh38'</code> | Genome assembly; GRCh37 routes to grch37.rest.ensembl.org |
| <code>annotations</code> | <code>EnsemblVEPAnnotationConfig</code> | <code>EnsemblVEPAnnotationConfig(verbose=0, device='cpu', timeout=3600, seed=None, canonical=False, hgvs=False, protein=False, domains=False, numbers=False, variant_class=False, uniprot=False, xref_refseq=False, mirna=False, pubmed=False, conservation=False, phenotypes=False, blosum62=False, max_ent_scan=False)</code> | Optional VEP annotations to include in the response |
| <code>sift</code> | <code>Literal['b', 'p', 's'] &#124; None</code> | <code>None</code> | SIFT output level: b=both, p=prediction, s=score; None = API default |
| <code>polyphen</code> | <code>Literal['b', 'p', 's'] &#124; None</code> | <code>None</code> | PolyPhen output level: b=both, p=prediction, s=score; None = API default |
| <code>mane</code> | <code>bool</code> | <code>False</code> | Include MANE Select annotations (GRCh38 only) |
| <code>alphamissense</code> | <code>bool</code> | <code>False</code> | AlphaMissense missense pathogenicity scores (human only) |
| <code>revel</code> | <code>bool</code> | <code>False</code> | REVEL ensemble pathogenicity scores (human only) |
| <code>cadd</code> | <code>bool</code> | <code>False</code> | CADD deleteriousness scores (human only) |
| <code>appris</code> | <code>bool</code> | <code>False</code> | APPRIS principal-isoform tag per transcript (human/mouse only) |
| <code>tsl</code> | <code>bool</code> | <code>False</code> | Transcript support level (TSL) per transcript (human/mouse only) |
| <code>ccds</code> | <code>bool</code> | <code>False</code> | Include CCDS identifier per transcript (human/mouse only — CCDS is a human/mouse project) |
| <code>distance</code> | <code>int &#124; None</code> | <code>None</code> | Bases up/downstream considered for consequence terms (None = API default 5000) |
| <code>pick</code> | <code>bool</code> | <code>False</code> | Collapse output to one consequence per variant via Ensembl PICK heuristic |
| <code>per_gene</code> | <code>bool</code> | <code>False</code> | Collapse output to one consequence per gene; incompatible with pick |

In [21]:
# Display output docs
display_api_reference("ensembl", "output", "run_ensembl_vep")

**Output** — `EnsemblVEPOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>consequences</code> | <code>list[EnsemblVEPConsequence]</code> | <code>[]</code> | One record per VEP input (Ensembl returns a list even for a single HGVS) |
| <code>source_url</code> | <code>str</code> | required | Final Ensembl REST URL that was hit |
| <code>raw_payload</code> | <code>list[dict[str, Any]]</code> | <code>[]</code> | Raw API JSON returned by Ensembl |

**`EnsemblVEPConsequence`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>input</code> | <code>str</code> | required | HGVS notation string the API echoed back |
| <code>most_severe_consequence</code> | <code>str</code> | required | Highest-severity Sequence Ontology consequence term |
| <code>seq_region_name</code> | <code>str &#124; None</code> | <code>None</code> | Chromosome or contig name |
| <code>start</code> | <code>int &#124; None</code> | <code>None</code> | 1-indexed inclusive genomic start coordinate |
| <code>end</code> | <code>int &#124; None</code> | <code>None</code> | 1-indexed inclusive genomic end coordinate |
| <code>strand</code> | <code>int &#124; None</code> | <code>None</code> | Genomic strand (+1 or -1) |
| <code>allele_string</code> | <code>str &#124; None</code> | <code>None</code> | Reference/alternate alleles (e.g. 'G/C') |
| <code>transcript_consequences</code> | <code>list[dict[str, Any]]</code> | <code>[]</code> | Per-transcript consequence records as raw dicts (shape varies by record type) |
| <code>colocated_variants</code> | <code>list[dict[str, Any]]</code> | <code>[]</code> | Co-located known variants (rsIDs, frequencies, clinical significance) |

In [22]:
vep = run_ensembl_vep(EnsemblVEPInput(hgvs="9:g.22125504G>C"))
top = vep.consequences[0]
print(f"{top.allele_string}: {top.most_severe_consequence}, {len(top.transcript_consequences)} transcripts affected")

G/C: intron_variant, 22 transcripts affected


## GRCh37 — same gene, different coordinates

Set `assembly="GRCh37"` for legacy/clinical datasets pinned to the older assembly.

In [23]:
grch37 = run_ensembl_lookup(
    EnsemblLookupInput(ensembl_id=gene.id),
    EnsemblLookupConfig(assembly="GRCh37"),
)
print(f"GRCh38 start: {gene.start} -> GRCh37 start: {grch37.result.start}")

GRCh38 start: 43044292 -> GRCh37 start: 41196312
